# Test Models

This notebook loads the `mlp_baseline` and `cnn_baseline` checkpoints from `final_models/` and evaluates them on the `test` splits.

- `mlp_baseline` uses the `LandmarksDataset` with `flatten=True`.
- `cnn_baseline` uses the `RGBImageDataset`.

Metrics reported (printed side-by-side): Top-1 Accuracy, Macro F1, Macro Precision, Macro Recall, per-class classification report, and confusion matrices (plotted side-by-side).

Results are printed only (not saved). You can add more models later by adding their name and checkpoint paths in the `models_to_test` list in the code cell below.

In [8]:
# Imports and setup
import os
import sys
from typing import Dict, Any, Tuple

import torch
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import itertools

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

# Project root (adjust if needed)
PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

# Import model factories and datasets from repo
from model_classes.mlp_baseline import create_mlp_baseline
from model_classes.cnn_baseline import create_cnn_baseline
from data_loaders import LandmarksDataset, RGBImageDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [9]:
# Helper: robustly extract model state dict from saved checkpoint
def extract_state_dict(ckpt: Any) -> dict:
    # ckpt may be a pure state_dict or a wrapper with keys like 'state', 'model_state', 'state_dict'
    if isinstance(ckpt, dict):
        # common patterns
        if 'state' in ckpt and isinstance(ckpt['state'], dict):
            s = ckpt['state']
            if 'model_state' in s:
                return s['model_state']
            if 'state_dict' in s:
                return s['state_dict']
        if 'model_state' in ckpt and isinstance(ckpt['model_state'], dict):
            return ckpt['model_state']
        if 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
            return ckpt['state_dict']
        # Might be full state dict already
        # Heuristic: many values are tensors -> assume this is the state_dict
        # Ensure keys look like parameter names (contain '.')
        if all(isinstance(k, str) and '.' in k for k in ckpt.keys()):
            return ckpt
    raise RuntimeError('Unable to extract model state dict from checkpoint')

# Evaluation routine that collects predictions and computes metrics
def evaluate_model(model: torch.nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, Any]:
    model = model.to(device).eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            # support both landmark and rgb dataset dict formats
            if isinstance(batch, dict):
                # Landmark dataset uses key 'landmarks' ; RGB uses 'image'
                if 'landmarks' in batch:
                    x = batch['landmarks'].to(device)
                elif 'image' in batch:
                    x = batch['image'].to(device)
                else:
                    raise RuntimeError('Unknown batch format')
                y = batch['label']
            else:
                x, y = batch
            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()
            labels = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.array(y)
            all_preds.append(preds)
            all_labels.append(labels)
    if not all_preds:
        return {}
    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_labels, axis=0)

    acc = float(accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    prec_macro = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    rec_macro = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'precision_macro': prec_macro,
        'recall_macro': rec_macro,
        'confusion_matrix': cm,
        'classification_report': report,
        'y_true': y_true,
        'y_pred': y_pred,
    }

# Small helper to print summary succinctly
def print_summary(name: str, metrics: Dict[str, Any]):
    print('--- {} ---'.format(name))
    print(f"Top-1 Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro F1: {metrics['f1_macro']:.4f}")
    print(f"Macro Precision: {metrics['precision_macro']:.4f}")
    print(f"Macro Recall: {metrics['recall_macro']:.4f}")
    print('Classification Report:')
    print(metrics['classification_report'])

def plot_confusion_matrix(cm, class_names, ax=None, title='Confusion Matrix'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title)
    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=90, fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, format(cm[i, j], fmt),
                horizontalalignment='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=5)
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')
    return ax

In [10]:
# Configuration: hard-coded checkpoint paths (update later to add more models)
final_models_dir = os.path.join(PROJECT_ROOT, 'final_models')
# Hard-coded filenames as requested
mlp_ckpt_path = os.path.join(final_models_dir, 'mlp_baseline.pth')
cnn_ckpt_path = os.path.join(final_models_dir, 'cnn_baseline.pth')

# Prepare datasets (test split) and loaders
print('Preparing LandmarksDataset (flatten=True) on test split...')
lm_ds = LandmarksDataset(annotations_root=os.path.join(PROJECT_ROOT, 'data', 'annotations'), split='test', flatten=True)
print(lm_ds)
lm_loader = DataLoader(lm_ds, batch_size=64, shuffle=False)

print('Preparing RGBImageDataset on test split...')
rgb_ds = RGBImageDataset(crops_root=os.path.join(PROJECT_ROOT, 'data', 'crops'), split='test')
print(rgb_ds)
rgb_loader = DataLoader(rgb_ds, batch_size=64, shuffle=False)

# Models to test list: (name, factory_fn, ckpt_path, dataset, loader)
models_to_test = [
    ('mlp_baseline', create_mlp_baseline, mlp_ckpt_path, lm_ds, lm_loader),
    ('cnn_baseline', create_cnn_baseline, cnn_ckpt_path, rgb_ds, rgb_loader),
]

Preparing LandmarksDataset (flatten=True) on test split...
LandmarksDataset(split='test', size=89047, num_classes=18, flatten=True)
Preparing RGBImageDataset on test split...
RGBImageDataset(split='test', size=90003, num_classes=18)
LandmarksDataset(split='test', size=89047, num_classes=18, flatten=True)
Preparing RGBImageDataset on test split...
RGBImageDataset(split='test', size=90003, num_classes=18)


In [11]:
# Run evaluations for the configured models and collect results
results = {}
for name, factory, ckpt_path, ds, loader in models_to_test:
    print(f'\nEvaluating {name}...')
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path} -- skipping {name}')
        continue
    # Instantiate model with dataset-derived shapes where possible
    try:
        if name.startswith('mlp'):
            # infer flattened input dim from one sample
            sample = ds[0]['landmarks']
            input_dim = int(sample.numel()) if hasattr(sample, 'numel') else int(np.prod(sample.shape))
            model = factory(input_dim=input_dim, num_classes=ds.num_classes)
        else:
            # CNN: infer num_classes only
            model = factory(num_classes=ds.num_classes)
    except Exception as e:
        print('Failed to instantiate model for', name, '->', e)
        continue

    # Load checkpoint (robustly)
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        try:
            state = extract_state_dict(ckpt)
        except RuntimeError:
            # maybe checkpoint was a plain state dict saved directly
            if isinstance(ckpt, dict) and all(isinstance(k, str) and '.' in k for k in ckpt.keys()):
                state = ckpt
            else:
                raise
        model.load_state_dict(state)
    except Exception as e:
        print('Failed to load checkpoint for', name, '->', e)
        continue

    # Evaluate
    try:
        metrics = evaluate_model(model, loader, device)
        if not metrics:
            print('No predictions for', name)
            continue
        results[name] = { 'metrics': metrics, 'class_names': ds.class_names() }
        print(f'Done evaluating {name}')
    except Exception as e:
        print('Evaluation failed for', name, '->', e)

# If no results found notify user
if not results:
    print('No model results available. Check that checkpoint files exist in final_models/ and datasets are available under data/.')


Evaluating mlp_baseline...
Failed to load checkpoint for mlp_baseline -> Error(s) in loading state_dict for MLPBaseline:
	Missing key(s) in state_dict: "backbone.3.weight", "backbone.3.bias". 
	Unexpected key(s) in state_dict: "backbone.1.weight", "backbone.1.bias", "backbone.1.running_mean", "backbone.1.running_var", "backbone.1.num_batches_tracked", "backbone.4.weight", "backbone.4.bias", "backbone.5.weight", "backbone.5.bias", "backbone.5.running_mean", "backbone.5.running_var", "backbone.5.num_batches_tracked". 
	size mismatch for backbone.0.weight: copying a param with shape torch.Size([256, 42]) from checkpoint, the shape in current model is torch.Size([128, 42]).
	size mismatch for backbone.0.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for head.weight: copying a param with shape torch.Size([18, 128]) from checkpoint, the shape in current model is torch.Size([18, 64]).

Evaluating cnn_baselin

In [12]:
# Print summaries side-by-side for quick comparison
if results:
    # Print a compact table of core metrics
    names = list(results.keys())
    print('=== Summary Table ===')
    header = ['Model', 'Top-1 Acc', 'Macro F1', 'Macro Prec', 'Macro Rec']
    rows = []
    for n in names:
        m = results[n]['metrics']
        rows.append([n, f"{m['accuracy']:.4f}", f"{m['f1_macro']:.4f}", f"{m['precision_macro']:.4f}", f"{m['recall_macro']:.4f}"])
    # Print table-like output
    print(' | '.join(header))
    for r in rows:
        print(' | '.join(r))

    # Print full classification reports side-by-side (one after another)
    for n in names:
        print('\n' + '='*40)
        print(f'Results for: {n}')
        print_summary(n, results[n]['metrics'])

    # Plot confusion matrices side-by-side
    num = len(names)
    fig, axes = plt.subplots(1, num, figsize=(6 * num, 6))
    if num == 1:
        axes = [axes]
    for ax, n in zip(axes, names):
        cm = results[n]['metrics']['confusion_matrix']
        classes = results[n]['class_names']
        plot_confusion_matrix(cm, classes, ax=ax, title=n)
    plt.tight_layout()
    plt.show()
else:
    print('No results to display.')

No results to display.
